# Étape 3 — Clustering

**Instance** : un couple (année, point de grille) — les 12 valeurs mensuelles de
`t2m` et du rayonnement thermique net `str`, plus les features de tendance
(intra-annuelles et long terme).

Algorithmes appliqués **année par année** : K-Means, DBSCAN, AGNES, AutoEncoder + K-Means.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)

In [ ]:
from src import clustering as cl
from src import config as C

clean = pd.read_parquet(C.FILE_CLEAN)
profiles = cl.build_year_profiles(clean)
profiles.head()

In [ ]:
cols = cl.feature_columns(profiles)
print(f"{len(cols)} features :")
print([c for c in cols if not c[-2:].isdigit()])

## 1. Les features de tendance long terme

`_anomalie` : écart de l'année à la normale du point (5 premières années).
`_derive` : pente des moyennes annuelles du point sur toute la période.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
profiles.groupby("year")["t2m_anomalie"].mean().plot(marker="o", ax=axes[0],
    title="Anomalie moyenne de t2m (°C)")
axes[0].axhline(0, color="grey", lw=1)
der = profiles.drop_duplicates(["latitude", "longitude"])
s = axes[1].scatter(der["longitude"], der["latitude"], c=der["t2m_derive"],
                    cmap="RdYlBu_r", s=18)
plt.colorbar(s, ax=axes[1], label="°C/an")
axes[1].set_title("Dérive pluriannuelle de t2m par point")
plt.tight_layout();

## 2. Choix de k

In [ ]:
scores_k = cl.elbow(profiles)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
scores_k.plot(x="k", y="inertie", marker="o", ax=axes[0], title="Coude (inertie)")
scores_k.plot(x="k", y="silhouette", marker="o", ax=axes[1], title="Silhouette")
plt.tight_layout();

## 3. Les quatre algorithmes, année par année

La standardisation est **globale** (et non par année) pour que les clusters restent comparables d'une année à l'autre. Les étiquettes sont renumérotées par température croissante : sans cela elles seraient arbitraires et l'analyse d'évolution impossible.

In [ ]:
labels, scores = cl.run_all_years(profiles)

In [ ]:
resume = scores.groupby("algo")[["n_clusters", "bruit", "silhouette",
                                  "calinski", "davies_bouldin"]].mean()
resume.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m, titre in zip(axes,
                        ["silhouette", "calinski", "davies_bouldin"],
                        ["Silhouette (↑)", "Calinski-Harabasz (↑)", "Davies-Bouldin (↓)"]):
    sns.lineplot(data=scores, x="year", y=m, hue="algo", marker="o", ax=ax)
    ax.set_title(titre)
plt.tight_layout();

> **Lecture de DBSCAN.** DBSCAN ne cherche pas k groupes : il cherche des zones
> denses séparées par du vide. Si le nuage est un continuum (gradient nord-sud
> régulier), il renvoie une ou deux classes et beaucoup de bruit — ce n'est pas
> un bug mais un résultat : il n'y a pas de frontière de densité. Comparer sa
> silhouette à celle de K-Means n'a alors guère de sens, les deux ne répondent
> pas à la même question.

## 4. Carte des clusters

Comparaison des quatre algorithmes sur la dernière année.

In [ ]:
derniere = labels["year"].max()
sub = labels[labels["year"] == derniere]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, algo in zip(axes.ravel(), ["kmeans", "agnes", "dbscan", "ae_kmeans"]):
    sns.scatterplot(data=sub, x="longitude", y="latitude", hue=algo,
                    palette="tab10", s=22, ax=ax, legend="brief")
    ax.set_title(f"{algo} — {derniere}")
plt.tight_layout();

## 5. Profils mensuels moyens par cluster

In [ ]:
merged = labels.merge(profiles, on=cl.KEYS)
mois = [f"t2m_{m:02d}" for m in range(1, 13)]
prof = merged[merged["kmeans"] >= 0].groupby("kmeans")[mois].mean().T
prof.index = range(1, 13)
prof.plot(figsize=(9, 4), marker="o", xlabel="mois", ylabel="t2m (°C)",
          title="Profil annuel moyen par cluster (K-Means)");

## 6. Évolution des clusters sur la décennie

In [ ]:
evo = cl.evolution(labels, profiles, algo="kmeans")
evo.round(2).head(12)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
evo.pivot(index="year", columns="cluster", values="part").plot.area(
    ax=axes[0], title="Part des points par cluster")
evo.pivot(index="year", columns="cluster", values="t2m_moyen").plot(
    marker="o", ax=axes[1], title="Température moyenne par cluster (°C)")
evo.pivot(index="year", columns="cluster", values="anomalie").plot(
    marker="o", ax=axes[2], title="Anomalie moyenne par cluster (°C)")
axes[2].axhline(0, color="grey", lw=1)
plt.tight_layout();

Matrice de transition : où sont passés, en fin de période, les points de chaque cluster de départ.

In [ ]:
trans = cl.transitions(labels, algo="kmeans")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(trans, annot=True, fmt=".2f", cmap="Blues", ax=ax)
ax.set_title(f"Transitions {labels['year'].min()} → {labels['year'].max()} (K-Means)");